CNN PER OBJECT DETECTION (YOLO, R-CNN-PANORAMICA)

Classificazione del cosa e localizzazione del dove

Fino ad ora abbiamo insegnato a fare una sola cosa (gaurdare un immagine e dirci cosa contiene).
Passiamo dalla semplice vista alla consapevolezza spaziale.
non mi accontento di sapere che c'è un gatto ma voglio sapere dove si trovare disegnando un perimetro intorno.

- Dalla classificazione alla localizzazione
- Paradigmi a confronto, yolo e R-CNN
- Ingegneria della predizione, il concetto di Anchor box

Con l'Object Detection si alza l'asticella, si passa da un output come un semplice gatto/cane/automobile ma si cerca di trovare degli elementi all'interno dello spazio.
Non ci chiediamo più solo cosa è presente in una foto, ma chiediamo al modello di indicare 'dove' si trovano gli oggetti, permettendo di identificare più istanza in un singolo frame.
La rete, quidni, non deve più solo resituire una distribuzione di probabilità sulle classi, ma anche un set di coordinate numeriche che delimitano il perimetro di un oggetto rilevato.

Bounding Box e Regressione
Definire la geometrica del rilevamento.
Il cuore della localizzazione è il boundig box, un rettangolo immaginario che racchiude l'oggetto. Non contiene solo probabilità ma anche le coordinate per disegnare una cornice, più la rete è addestrata più le coordinate disegnano una cornice intorno all'oggetto reale.
- Localizzazione: il processo di calcolo delle coordinate spaziali di un rettangolo che racchiude l'oggetto di interesse
- Parametri del box: solitamente definiti da quattro valori ovvero centro x, centro y, larghezza e altezza, normalizzati rispetto alla immagine.
- Regressione della posizione: la rete neurale deve imparare a prevedere valori continui per queste coorinate minimizzando l'errore di distanza.
- La predizione finale combina il vettore della classi con il vettore della coordinate geometriche per ogni oggetto.

Ma come fa la rete a sapere se la cornice è disegnata nel vuoto o se racchiude davvero qualcosa?

Il Vettore di Predizione
Confidenza dell oggetto (il parametro pc):
Il parametri pc indica la probabilità che un oggetto di qualsiasi classe sia presente all'interno del box proposto dal modello.
Multi-Label Classification: 
A differenza della classificazione standard, qui possiamo avere più classi attive se gli oggetti di sovrappongono o se il dataset lo prevede. Esempio in quel punto c'è un veicolo ma anche un autobus
Normalizzazione Coordinate:
Le coordinate vengono espresso in 0 e 1 rispetto alle dimensioni dell'immagine per rendere il modello invarinte alla risoluzione di input.

Addestrare un sistema cos' richiede un bilanciamente delicato.

Loss Multitasking
Bilanciare classificazione e precisione spaziale
Addestrare un detector significa sommare due funzioni di costo diverse: una per la precisione della classe e una per la precisione del rettangolo.
Se non bilanciamo correttamente questi due pesi, rischiamo di avere una rete che riconosce bene l'oggetto ma sbaglia l'inquadratura o viceversa.
Usiamo quindi una loss multi tasking che somma questi due errori.

Strategia di Rilevamento
Evoluzione dai due stadi alla visione istantanea
Esistono due filosofie principali per risolvere l Object Detection. La prima separa la ricerca delle zone interessanti dalla classificazione (prima cerca dove potrebbe essere degli oggetti e poi li analizza per capire cosa sono, two stage), la seconda esegue tutto in un unico passaggio (guardiamo l'immagine una vota sola e capiamo tutto, one stage).
La scelta tra questi due paradigmi influenza drasticamente la velocità di esecuzione (FPS) e la capacità di rilevare oggetti molto piccoli o densi.

YOLO vs R-CNN
One-stage contro Two-stage
- Two-stage (R-CNN): propone prima delle regioni candidate (RPN) e poi le analizza una ad una. Alta precisione, ma computazionalmente pesante. Lenti ma quasi infallibili. Se deve leggere una risonanza medica, dove la precisione è tutto, R-CNN è la soluzione
- One-stage (Yolo): divide l'immagine in una griglia e ogni cella grida cosa vede e dove si trova l'oggetto.Prevede box e classi in un unico passaggio di feed-forward. Estramamente veloce. Se devo far volare un drone che evita oggetti, Yolo è l'unica scelta possibile
- Velocità di inferenza: i modelli one-stage sono ideali per applicazioni real-time come la guida autonoma o la videosorviglianza.

Il throughput dei modelli YOLO permette di elaborare video in tempo reale superando i limiti dei modelli regionali.

Meccanismo Interni
Region Proposals
Nel modello Two-stage, un algoritmo secondario setaccia l'immagine alla ricerca di 'macchie' che potrebbero contenere oggetti prima di interpellare la rete principale.
Griglia e Celle
Durante il training, un oggetto reale viene assegnato all anchor-box che ha l'IoU più alto con esso.
Backbone e Neck
Entrambi i modelli usano una CNN base (es. ResNet) per estrarre le feature (caratteristiche), ma differiscono nel modo in cui ricompongono i pezzi per la decisione finale (predizione).
Questo ci porta ad una scelta critica che ogni ingeniere deve compiere

Il Trade-off Velocità-Precisione
Scegliere il modello giusto per il problema
Non esiste il modello perfetto, esiste il modello giusto per ogni problema.
Se dobbiamo analizzare vetrini medici ad altissima risoluzione, la lentezza di una R-CNN è giustificata dalla precisione millimetrica richiesta.
Per un drone che deve evitare ostacoli in volo, la latenza di pochi millisecondi di Yolo è l'unico parametro che conta davvere per la sicurezza.

Ma come fa una cella quadrata a prevedere la forma di una persona o di un camion?

Anchor Boxes e IoU
Ottimizzare la sovrapposizione e le forme
Uno dei problemi più difficili sono le forme diverse
La soluzione risiede nella Anchor Box, ovvero modelli di forma predefiniti che aiutano la rete a specializzarsi su diverse proporzioni e scale dimensionali.
Non possiamo usare solo quadrati, saremmo in difficoltà.
Le Anchor Box hanno forma e grandezze predefinite (es. alte e strette per i pedoni)

Geometria delle Anchor
I vestiti su misura per gl oggetti.
- Anchor Boxes: rettangoli con aspect-ratio predefiniti che fungono da riferimento per la regressione della forme finale.
- Intersection over Union (IoU): metrica che misura quanto la zona prevista e la zona reale si sovrappongono.
- Non-Maximum Suppression (NMS): algoritmo che elimina i box duplicati o troppo simili, mantenendo solo quello con la confidenza più alta, eliminando gli altri.
- L'IoU è calcolato come il rapporto tra l'area dell'intersezione dei box e l'area della loro unione.
IoU= Area (A intersecato B / Area A unione B), rapporto tra l'area dove due oggetti si toccano, e l'area totale dei due oggetti insieme. Se IoU è vicino a 1 allora la sovrapposizione è perfetta se è zero abbiamo mancato completamente il bersaglio

Gestione delle Scale
Le Anchor Box permettono sia al modello di speciaizzarsi, ma  durante l'addestramento ogni oggetto reale viene assegnato alla anchor-box che gli assomiglia di èiù per forma e dimensioni.
Se una persona passa dantvi alla camera, la rete attiva la anchor box verticale.

Problema di Affollamento
Oltre il singolo rilevamento.
Senza Anchor Boxes, una singola cella della griglia farebbe fatica a prevedere due oggetti diversi che si trovano nello stesso punto.
L'uso di multiple anchor box risolve questo problema, permettendo al modello di distinguere, ad esempio, una persona che cammina davanti a un aouto parcheggiata.
Questo ha permesso ai detector modermi di gestire scene urbane con oggetti diversi.



In [1]:
def calculate_iou(boxA, boxB):
    """
    Calcola l'Intersection over Union (IoU) tra due bounding box.
    
    Formato input atteso: [x1, y1, x2, y2]
    (x1, y1) -> Coordinate del vertice in alto a sinistra (Top-Left)
    (x2, y2) -> Coordinate del vertice in basso a destra (Bottom-Right)
    """
    
    # 1. DETERMINAZIONE DEL RETTANGOLO INTERSEZIONE
    # L'intersezione di due rettangoli è essa stessa un rettangolo.
    # Per trovare il vertice in alto a sinistra dell'intersezione, prendiamo il MASSIMO tra le x1 e le y1.
    # Per trovare il vertice in basso a destra, prendiamo il MINIMO tra le x2 e le y2.
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    # 2. CALCOLO DELL'AREA DI INTERSEZIONE
    # Calcoliamo larghezza (xB - xA) e altezza (yB - yA) dell'area comune.
    # Aggiungiamo "+ 1" perché le coordinate dei pixel sono inclusive (es. da 0 a 9 sono 10 pixel).
    # Il max(0, ...) è fondamentale: se i box non si sovrappongono, la sottrazione darebbe 
    # un numero negativo. Forzando a 0, l'area di intersezione sarà correttamente nulla.
    interWidth = max(0, xB - xA + 1)
    interHeight = max(0, yB - yA + 1)
    interArea = interWidth * interHeight

    # 3. CALCOLO DELLE AREE DEI SINGOLI BOX
    # Calcoliamo l'area occupata da ciascun rettangolo separatamente.
    # Usiamo la stessa logica (lato + 1) per coerenza con il calcolo dell'intersezione.
    boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)

    # 4. CALCOLO DELL'UNIONE E DELL'IOU
    # L'Unione non è semplicemente AreaA + AreaB, perché l'intersezione verrebbe contata due volte.
    # La formula corretta è: Area(A ∪ B) = Area(A) + Area(B) - Area(A ∩ B)
    unionArea = float(boxAArea + boxBArea - interArea)
    
    # L'IoU è il rapporto tra quanto i box "condividono" e quanto spazio "occupano insieme".
    # Il valore è sempre compreso tra 0 (nessuna sovrapposizione) e 1 (sovrapposizione perfetta).
    iou = interArea / unionArea

    return iou

# --- TEST E ANALISI ---
# Ground truth: la posizione corretta dell'oggetto nel dataset
ground_truth = [50, 50, 150, 150]
# Prediction: la scatola disegnata dal nostro modello di Deep Learning
prediction = [60, 60, 170, 160]

result = calculate_iou(ground_truth, prediction)
print(f"Intersection over Union: {result:.4f}")

Intersection over Union: 0.6306
